# Scrape New SF Court Cases (Local)

Find new small claims cases by **probing case-number ranges** and add them to
`valid_cases.json`. **Run this on your laptop, not in Colab** — Cloudflare
blocks Colab IPs.

### What this does
1. Loads existing `valid_cases.json` so we don't re-probe known cases
2. Asks for a SessionID (manual — Cloudflare blocks automation here)
3. Starts a background keepalive thread to help extend the session
4. Probes every case number in the configured range via `GetROA`
5. Saves valid cases to `valid_cases.json` incrementally — interrupt-safe

### Why probing instead of dates?
The court calendar API (`GetCases2`) only returns recent dates — anything more
than a couple months old comes back empty. The `GetROA` endpoint works for any
case that exists, regardless of age, so we brute-force a numeric range of
case numbers instead. The case-number prefix encodes year: `CSM25...` = 2025,
`CSM26...` = 2026.

In [1]:
import json
import time
from pathlib import Path

# ============================================================
# CONFIGURATION
# ============================================================
PROBE_PREFIX    = "CSM"           # case number prefix (CSM = small claims)
PROBE_START_NUM = 26871856        # inclusive — Phase C: extend 2026 above prior probed max (26871855)
PROBE_END_NUM   = 26873000        # inclusive — stop here unless hit rate stays high
PROBE_STEP      = 1               # dense probe — collect every existing case
PROBE_DELAY     = 1.0             # seconds between probes
# ============================================================

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
VALID_CASES_PATH = REPO_ROOT / "scraper" / "state" / "valid_cases.json"

BASE_URL = "https://webapps.sftc.org"
CALENDAR_PATH = "/cc/CaseCalendar.dll"  # used by keepalive
CASE_PATH     = "/ci/CaseInfo.dll"      # used by probing
COURT_URL = f"{BASE_URL}{CALENDAR_PATH}"
CASE_TYPE = "M//CSM"                    # used by keepalive
USER_AGENT = "MSDS603-Research-Scraper/1.0 (SF Small Claims academic study)"

total_in_range = len(range(PROBE_START_NUM, PROBE_END_NUM + 1, PROBE_STEP))
print(f"Repo root:        {REPO_ROOT}")
print(f"valid_cases.json: {VALID_CASES_PATH}")
print(f"Probe range:      {PROBE_PREFIX}{PROBE_START_NUM} → {PROBE_PREFIX}{PROBE_END_NUM} (step {PROBE_STEP})")
print(f"  ({total_in_range} candidates)")

Repo root:        /Users/ernesto/Desktop/MLOPS-Project/litigation-outcome-pipeline
valid_cases.json: /Users/ernesto/Desktop/MLOPS-Project/litigation-outcome-pipeline/scraper/state/valid_cases.json
Probe range:      CSM26871856 → CSM26873000 (step 1)
  (1145 candidates)


## Step 1 — Load existing valid_cases.json

Reads the cases we've already discovered so we can skip them. Anything already
in `probed` or `valid` will NOT be re-fetched.

In [2]:
if VALID_CASES_PATH.exists():
    store = json.loads(VALID_CASES_PATH.read_text())
    existing_cases = set(store.get("valid", {}).keys())
    already_probed = set(store.get("probed", []))
    print(f"✓ Loaded {VALID_CASES_PATH.name}")
    print(f"  Already-known valid cases: {len(existing_cases)}")
    print(f"  Already-probed (any result): {len(already_probed)}")
else:
    store = {"valid": {}, "not_found": [], "probed": []}
    existing_cases = set()
    already_probed = set()
    print(f"⚠ {VALID_CASES_PATH.name} does not exist — starting fresh")

✓ Loaded valid_cases.json
  Already-known valid cases: 2664
  Already-probed (any result): 6490


## Step 2 — Get a SessionID and start the keepalive

The court site is gated by Cloudflare's "verify you're a person" check, so the
very first step needs a human. You solve the verification once in your browser
and paste the SessionID here.

After that, a background thread pings the court API every 60 seconds. Note: the
site likely uses a hard session TTL (not idle), so pings may not fully prevent
expiry — the probing loop in Step 3 handles re-prompting gracefully if needed.

**To get a SessionID:**
1. Open https://webapps.sftc.org/cc/CaseCalendar.dll in your browser
2. Solve the Cloudflare verification (one click)
3. Once the calendar loads, copy the hex string after `SessionID=` in the URL bar

In [ ]:
import threading
import requests
import subprocess
from datetime import date as _date

# Mutable holder so the keepalive thread can pick up new sessions if expiry happens
_session_holder = {"id": ""}
_keepalive_started = {"flag": False}


def alert(msg="Session ID expired. Paste new one."):
    """Audible spoken alert so the user can step away from the keyboard."""
    try:
        subprocess.Popen(["say", msg])
    except Exception:
        pass


def start_keepalive(session_id: str, ping_interval: int = 60) -> None:
    """Start (or update) a daemon thread that pings the court API every
    `ping_interval` seconds. Idempotent — calling again with a new session_id
    just updates the holder; only one thread ever runs per kernel."""
    _session_holder["id"] = session_id
    if _keepalive_started["flag"]:
        print("  ↻ Keepalive updated with new SessionID")
        return
    _keepalive_started["flag"] = True

    def ping_loop():
        while True:
            sid = _session_holder["id"]
            if sid:
                try:
                    requests.get(
                        f"{BASE_URL}{CALENDAR_PATH}/datasnap/rest/TServerMethods1/"
                        f"GetCases2/{_date.today().isoformat()}/{CASE_TYPE}/{sid}",
                        headers={"User-Agent": USER_AGENT},
                        timeout=10,
                    )
                except Exception:
                    pass
            time.sleep(ping_interval)

    threading.Thread(target=ping_loop, daemon=True).start()
    print(f"  ✓ Keepalive thread started (pinging every {ping_interval}s)")


print("=" * 60)
print("SESSION ID NEEDED")
print("=" * 60)
print(f"\n1. Open this URL in your browser:")
print(f"   {COURT_URL}")
print(f"\n2. Solve the Cloudflare verification (click the checkbox)")
print(f"\n3. After the calendar loads, copy the hex string after `SessionID=`")
print(f"   in the URL bar. Example:")
print(f"     https://...?SessionID=A1B2C3D4...  →  copy A1B2C3D4...")
print()

SESSION_ID = input("Paste SessionID: ").strip()
print(f"\n✓ Using SessionID: {SESSION_ID[:12]}... ({len(SESSION_ID)} chars)")

start_keepalive(SESSION_ID)

## Step 3 — Probe case numbers in the configured range

For each case number in `PROBE_START_NUM` to `PROBE_END_NUM`, hits:
```
GET /ci/CaseInfo.dll/datasnap/rest/TServerMethods1/GetROA/{case_num}/{SessionID}/
```

The response is `{"result": [count, "[{...ROA entries...}]"]}`:
- `count == -1` → session expired (loop refreshes and retries the same case)
- `count == 0` → case doesn't exist (mark probed, move on)
- otherwise → case exists, save to `valid_cases.json`

Already-probed cases (from previous runs) are skipped automatically.
Progress is saved every 25 probes — safe to interrupt at any time.

In [ ]:
import subprocess


def alert(msg="Session ID expired. Paste new one."):
    """Audible spoken alert so the user can step away from the keyboard."""
    try:
        subprocess.Popen(["say", msg])
    except Exception:
        pass


def probe_case(case_num: str, sid: str):
    """Probe one case via GetROA. Returns (status, doc_count).
    status: 'expired' | 'found' | 'not_found' | 'error'"""
    url = f"{BASE_URL}{CASE_PATH}/datasnap/rest/TServerMethods1/GetROA/{case_num}/{sid}/"
    try:
        resp = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=20)
        resp.raise_for_status()
        data = resp.json()
        code = data["result"][0]
        if code == -1:
            return "expired", 0
        if code == 0:
            return "not_found", 0
        roa = json.loads(data["result"][1])
        return "found", len(roa)
    except Exception:
        return "error", 0


def save_progress() -> None:
    """Atomic-ish write of the in-memory store to disk."""
    VALID_CASES_PATH.parent.mkdir(parents=True, exist_ok=True)
    VALID_CASES_PATH.write_text(json.dumps(store, indent=2))


# Build target list, skipping already-probed cases (resume support)
candidates = [
    f"{PROBE_PREFIX}{n}"
    for n in range(PROBE_START_NUM, PROBE_END_NUM + 1, PROBE_STEP)
    if f"{PROBE_PREFIX}{n}" not in already_probed
]

print(f"Range:           {PROBE_PREFIX}{PROBE_START_NUM} → {PROBE_PREFIX}{PROBE_END_NUM} (step {PROBE_STEP})")
print(f"  Total in range:  {total_in_range}")
print(f"  Already probed:  {total_in_range - len(candidates)}")
print(f"  To probe:        {len(candidates)}")
print(f"  At {PROBE_DELAY}s/probe, expect ~{len(candidates) * PROBE_DELAY / 60:.1f} min")
print(f"  Saved every 25 probes — safe to interrupt.\n")

probed = 0
found = 0
not_found = 0
errors = 0
SAVE_EVERY = 25

for i, case_num in enumerate(candidates, 1):
    while True:  # retry loop for session expiry
        time.sleep(PROBE_DELAY)
        status, doc_count = probe_case(case_num, SESSION_ID)

        if status == "expired":
            save_progress()
            alert()
            print(f"  [{i}/{len(candidates)}] {case_num}: session expired (saved {found} found so far)")
            print("  → Get a fresh SessionID from your browser (refresh the court URL)")
            SESSION_ID = input("  Paste new SessionID: ").strip()
            start_keepalive(SESSION_ID)
            continue  # retry same case

        if status == "found":
            found += 1
            store.setdefault("valid", {})[case_num] = doc_count
            if case_num not in store.get("probed", []):
                store.setdefault("probed", []).append(case_num)
            already_probed.add(case_num)
            print(f"  [{i}/{len(candidates)}] {case_num}: FOUND ({doc_count} ROA entries)")
        elif status == "not_found":
            not_found += 1
            if case_num not in store.get("not_found", []):
                store.setdefault("not_found", []).append(case_num)
            if case_num not in store.get("probed", []):
                store.setdefault("probed", []).append(case_num)
            already_probed.add(case_num)
            print(f"  [{i}/{len(candidates)}] {case_num}: not found")
        else:
            errors += 1
            print(f"  [{i}/{len(candidates)}] {case_num}: ERROR")

        probed += 1
        break

    if i % SAVE_EVERY == 0:
        save_progress()

save_progress()

print("\n" + "=" * 50)
print("SUMMARY")
print("=" * 50)
print(f"  Probed this run:  {probed}")
print(f"  Found (NEW):      {found}")
print(f"  Not found:        {not_found}")
print(f"  Errors:           {errors}")
print(f"  Total valid now:  {len(store.get('valid', {}))}")
print(f"  Saved to:         {VALID_CASES_PATH}")

## Step 4 — Test GetROA vs GetDocuments for closed CSM24 recovery

**Hypothesis**: For closed cases, `GetDocuments` only returns disposition docs
(judgment, notice of entry, stipulation), but `GetROA` may return the full
historical docket including the original `CLAIM_OF_PLAINTIFF` SC-100.

If true, we can write a follow-up scraper that uses ROA-discovered doc IDs to
download claim PDFs for the 1,500+ CSM24 cases that are currently unrecoverable
(their claim docs are missing → no features → can't enter `dataset.csv`).

This cell tests 5 sample closed CSM24 cases:
- Calls `GetROA` and `GetDocuments` for each
- Diffs the descriptions
- Flags any **CLAIM_OF_PLAINTIFF** or **DEFENDANT_S_CLAIM** entries that show
  up in ROA but NOT in GetDocuments

Reuses the `SESSION_ID` already prompted for in Step 2. If your session expired,
re-run Step 2 first.

In [ ]:
import subprocess


def alert(msg="Session ID expired. Paste new one."):
    """Audible spoken alert so the user can step away from the keyboard."""
    try:
        subprocess.Popen(["say", msg])
    except Exception:
        pass


TEST_CASES = [
    "CSM24868105",  # dismissal docs only
    "CSM24868381",  # judgment + notice
    "CSM24868226",  # judgment + notice
    "CSM24868934",  # dismissal docs only
    "CSM24867823",  # judgment + notice + stipulation
]


def get_roa_entries(case_num: str, sid: str) -> list[dict] | None:
    url = f"{BASE_URL}{CASE_PATH}/datasnap/rest/TServerMethods1/GetROA/{case_num}/{sid}/"
    try:
        resp = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=20)
        resp.raise_for_status()
        data = resp.json()
        code = data["result"][0]
        if code == -1:
            return None  # session expired
        if code == 0:
            return []
        return json.loads(data["result"][1])
    except Exception as e:
        print(f"  ROA error: {e}")
        return []


def get_documents_entries(case_num: str, sid: str) -> list[dict] | None:
    url = f"{BASE_URL}{CASE_PATH}/datasnap/rest/TServerMethods1/GetDocuments/{case_num}/{sid}/"
    try:
        resp = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=20)
        resp.raise_for_status()
        data = resp.json()
        code = data["result"][0]
        if code == -1:
            return None  # session expired
        if code == 0:
            return []
        return json.loads(data["result"][1])
    except Exception as e:
        print(f"  Documents error: {e}")
        return []


def _claim_marker(text: str) -> str:
    upper = text.upper()
    for needle in ("CLAIM_OF_PLAINTIFF", "DEFENDANT_S_CLAIM", "SC-100",
                   "PLAINTIFF'S CLAIM", "PLAINTIFFS CLAIM"):
        if needle in upper:
            return " ⭐ CLAIM"
    return ""


found_any_claim = False

for case in TEST_CASES:
    print(f"\n{'='*70}\n{case}")
    roa = get_roa_entries(case, SESSION_ID)
    docs = get_documents_entries(case, SESSION_ID)

    if roa is None or docs is None:
        alert()
        print("  Session expired — re-run Step 2 with a fresh SessionID, then rerun this cell.")
        break

    print(f"  GetROA entries:       {len(roa)}")
    print(f"  GetDocuments entries: {len(docs)}")

    if roa:
        print(f"  Sample ROA entry keys: {list(roa[0].keys())}")
        sample = json.dumps(roa[0], indent=2)
        print(f"  Sample ROA entry: {sample[:500]}{'...' if len(sample) > 500 else ''}")

    # Try common description field names
    def _desc(entry: dict) -> str:
        for key in ("DESCRIPTION", "Description", "DOCUMENT_DESCRIPTION", "ENTRY",
                    "ENTRY_TEXT", "EntryText", "DOCKET_TEXT"):
            v = entry.get(key)
            if v:
                return str(v)
        return str(entry)

    roa_descs = {_desc(e) for e in roa}
    doc_descs = {_desc(e) for e in docs}

    extra = roa_descs - doc_descs
    print(f"  Entries in ROA but NOT in GetDocuments: {len(extra)}")
    for e in sorted(extra)[:15]:
        marker = _claim_marker(e)
        if marker:
            found_any_claim = True
        print(f"    - {e[:90]}{marker}")

print("\n" + "=" * 70)
if found_any_claim:
    print("⭐ CLAIM docs found in ROA — recovery path is viable.")
    print("   Next step: write a follow-up scraper that uses ROA-discovered doc IDs")
    print("   to download the claim PDFs for closed CSM24 cases.")
else:
    print("No claim docs surfaced in ROA either. 2024 likely unrecoverable via this API.")
    print("Fall back to manual UI download or public records request if 2024 features are needed.")

## Step 5 — Recover claim PDFs for closed CSM24 cases

Step 4 confirmed that `GetROA` exposes claim-doc entries (`MONEY CLAIM OF PLAINTIFF`,
`SC-100`, etc.) with downloadable URLs for closed cases — entries that
`GetDocuments` no longer returns.

This cell walks every CSM24 case in `{DRIVE}/pdfs/` that's missing a claim PDF,
hits `GetROA`, parses `RTEXT` for claim entries, and downloads the PDFs into
`{DRIVE}/pdfs/` using the existing filename convention so the rest of the
pipeline picks them up automatically.

**Resume-safe**: skips cases that already have a claim PDF. Safe to Ctrl+C and rerun.

**Tunables in the next cell:**
- `RECOVER_LIMIT = None` — cap on cases (try 20 first to sanity-check)
- `RECOVER_RATE_S = 1.0` — seconds between ROA calls (polite default)
- `RECOVER_DRY_RUN = False` — set True to discover only, no downloads

**After this finishes**: run `colab_gpu_extraction.ipynb` Steps 2 + 5 (PyMuPDF
text extract + LLM feature extraction) on the new PDFs, sync back to Drive, then
rebuild `dataset.csv` and retrain.

In [5]:
import re
import subprocess


def alert(msg="Session ID expired. Paste new one."):
    """Audible spoken alert so the user can step away from the keyboard."""
    try:
        subprocess.Popen(["say", msg])
    except Exception:
        pass


# ============================================================
# RECOVERY CONFIGURATION
# ============================================================
RECOVER_LIMIT   = None   # int or None — cap for testing. Try 20 first.
RECOVER_RATE_S  = 1.0    # seconds between GetROA calls
RECOVER_DRY_RUN = False  # True = discover only, don't download
# ============================================================

# Drive location for output PDFs (where the rest of the pipeline reads from)
DRIVE_DIR = next(
    (Path.home() / "Library/CloudStorage").glob("GoogleDrive-*/My Drive/litigation-pipeline")
)
PDFS_DIR = DRIVE_DIR / "pdfs"
print(f"Drive PDFs dir: {PDFS_DIR}")

# Map RTEXT signals → filename doc-type (more specific patterns first).
CLAIM_PATTERNS = [
    (re.compile(r"DEFENDANT.{0,3}S CLAIM", re.IGNORECASE), "DEFENDANT_S_CLAIM"),
    (re.compile(r"MONEY CLAIM OF PLAINTIFF", re.IGNORECASE), "CLAIM_OF_PLAINTIFF"),
    (re.compile(r"CLAIM OF PLAINTIFF FILED", re.IGNORECASE), "CLAIM_OF_PLAINTIFF"),
    (re.compile(r"PLAINTIFF.?S CLAIM AND ORDER", re.IGNORECASE), "CLAIM_OF_PLAINTIFF"),
    (re.compile(r"\bSC-100\b", re.IGNORECASE), "CLAIM_OF_PLAINTIFF"),
]


def _classify_rtext(rtext: str) -> str | None:
    for pattern, doc_type in CLAIM_PATTERNS:
        if pattern.search(rtext or ""):
            return doc_type
    return None


def _has_claim_pdf(case_num: str) -> bool:
    for p in PDFS_DIR.glob(f"{case_num}_*.pdf"):
        name = p.name.upper()
        if "CLAIM_OF_PLAINTIFF" in name or "DEFENDANT_S_CLAIM" in name:
            return True
    return False


def _list_csm24_to_recover() -> list[str]:
    by_case: dict[str, list] = {}
    for p in PDFS_DIR.glob("CSM24*.pdf"):
        cn = p.name.split("_")[0]
        by_case.setdefault(cn, []).append(p)
    return sorted(cn for cn in by_case if not _has_claim_pdf(cn))


def _download_pdf(url: str, dest: Path, session: requests.Session) -> bool:
    """Download PDF from a signed court URL. Returns True on success."""
    try:
        resp = session.get(url, timeout=60, stream=True)
        resp.raise_for_status()
        if "pdf" not in resp.headers.get("content-type", "").lower():
            return False
        dest.parent.mkdir(parents=True, exist_ok=True)
        with open(dest, "wb") as f:
            for chunk in resp.iter_content(chunk_size=8192):
                f.write(chunk)
        return True
    except Exception:
        return False


def _get_roa(case_num: str, sid: str) -> list[dict] | None:
    """Returns list of ROA entry dicts. None signals session expiry."""
    url = f"{BASE_URL}{CASE_PATH}/datasnap/rest/TServerMethods1/GetROA/{case_num}/{sid}/"
    try:
        resp = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=20)
        resp.raise_for_status()
        data = resp.json()
        code = data["result"][0]
        if code == -1:
            return None
        if code == 0:
            return []
        return json.loads(data["result"][1])
    except Exception:
        return []


# Build worklist
cases = _list_csm24_to_recover()
total_eligible = len(cases)
if RECOVER_LIMIT:
    cases = cases[:RECOVER_LIMIT]

print(f"Eligible CSM24 cases without claim PDF: {total_eligible}")
print(f"Processing this run:                    {len(cases)}")
print(f"Rate:                                   {RECOVER_RATE_S}s between ROA calls")
print(f"Dry run:                                {RECOVER_DRY_RUN}")
print(f"Estimated time:                         ~{len(cases) * (RECOVER_RATE_S + 0.5) / 60:.0f} min\n")

download_session = requests.Session()
download_session.headers.update({"User-Agent": USER_AGENT})

stats = {
    "ROA_called": 0,
    "ROA_no_entries": 0,
    "ROA_errors": 0,
    "no_claim_in_ROA": 0,
    "claim_entries_found": 0,
    "downloads_succeeded": 0,
    "downloads_failed": 0,
}

current_sid = SESSION_ID  # local copy so re-prompt doesn't clobber Step 2's state

for i, case in enumerate(cases, 1):
    time.sleep(RECOVER_RATE_S)

    roa = _get_roa(case, current_sid)
    if roa is None:
        alert()
        print(f"\n[{i}/{len(cases)}] {case}: session expired")
        current_sid = input("Paste fresh SessionID: ").strip()
        if not current_sid:
            print("Aborting.")
            break
        start_keepalive(current_sid)
        roa = _get_roa(case, current_sid)
        if roa is None:
            alert()
            print("  Retry also expired — aborting.")
            break
    stats["ROA_called"] += 1

    if not roa:
        stats["ROA_no_entries"] += 1
        continue

    # Find claim-doc entries with URLs
    claim_entries: list[tuple[str, str, str]] = []
    for entry in roa:
        rtext = entry.get("RTEXT", "") or ""
        url = entry.get("URL", "") or ""
        doc_type = _classify_rtext(rtext)
        if doc_type and url:
            claim_entries.append((doc_type, url, rtext[:80]))

    if not claim_entries:
        stats["no_claim_in_ROA"] += 1
        if i % 25 == 0 or i <= 10:
            print(f"[{i}/{len(cases)}] {case}: no claim entries ({len(roa)} ROA total)")
        continue

    stats["claim_entries_found"] += len(claim_entries)

    # Dedupe by doc_type — one CLAIM_OF_PLAINTIFF per case
    seen: set[str] = set()
    for doc_type, url, snippet in claim_entries:
        if doc_type in seen:
            continue
        seen.add(doc_type)

        out_path = PDFS_DIR / f"{case}_{doc_type}.pdf"
        if out_path.exists():
            continue

        if RECOVER_DRY_RUN:
            print(f"[{i}/{len(cases)}] {case}: WOULD DOWNLOAD {doc_type}")
            continue

        if _download_pdf(url, out_path, download_session):
            stats["downloads_succeeded"] += 1
            kb = out_path.stat().st_size // 1024
            print(f"[{i}/{len(cases)}] {case}: ✓ {doc_type}.pdf ({kb} KB)")
        else:
            stats["downloads_failed"] += 1
            print(f"[{i}/{len(cases)}] {case}: ✗ {doc_type} download failed")

print("\n" + "=" * 60)
print("RECOVERY SUMMARY")
print("=" * 60)
for k, v in stats.items():
    print(f"  {k:25s} {v}")

if stats["downloads_succeeded"] > 0:
    print(f"\n{stats['downloads_succeeded']} new claim PDFs in {PDFS_DIR}")
    print("\nNext: colab_gpu_extraction.ipynb Steps 2 + 5, then build_training_rows.py")

Drive PDFs dir: /Users/ernesto/Library/CloudStorage/GoogleDrive-ernestod1998@gmail.com/My Drive/litigation-pipeline/pdfs
Eligible CSM24 cases without claim PDF: 956
Processing this run:                    956
Rate:                                   1.0s between ROA calls
Dry run:                                False
Estimated time:                         ~24 min


[1/956] CSM24867798: session expired
  ↻ Keepalive updated with new SessionID
[1/956] CSM24867798: no claim entries (2 ROA total)
[2/956] CSM24867801: no claim entries (6 ROA total)
[3/956] CSM24867821: no claim entries (11 ROA total)
[4/956] CSM24867822: no claim entries (11 ROA total)
[5/956] CSM24867826: no claim entries (9 ROA total)
[6/956] CSM24867831: no claim entries (4 ROA total)
[7/956] CSM24867844: no claim entries (9 ROA total)
[8/956] CSM24867852: no claim entries (9 ROA total)
[9/956] CSM24867866: no claim entries (12 ROA total)
[10/956] CSM24867874: no claim entries (8 ROA total)
[25/956] CSM24868136: no claim